In [1]:
import pandas as pd
import numpy as np

BASE_PATH = r'C:\Users\harsh\OneDrive\Desktop\ipl data analysis\datanew'
matches = pd.read_csv(rf'{BASE_PATH}\matches_clean.csv', parse_dates=['match_date'])
deliveries = pd.read_csv(rf'{BASE_PATH}\deliveries_clean.csv')

In [2]:
matches_ml = matches[(matches['match_winner'].notna()) & 
                     (matches['season'] != 2026)].copy()

team1 = matches_ml[['match_id', 'season', 'match_date', 'venue',
                     'team1', 'team2', 'toss_winner', 'match_winner']].copy()
team1['team'] = team1['team1']
team1['opponent'] = team1['team2']
team1['won'] = (team1['match_winner'] == team1['team1']).astype(int)

team2 = matches_ml[['match_id', 'season', 'match_date', 'venue',
                     'team1', 'team2', 'toss_winner', 'match_winner']].copy()
team2['team'] = team2['team2']
team2['opponent'] = team2['team1']
team2['won'] = (team2['match_winner'] == team2['team2']).astype(int)

base = pd.concat([team1, team2], ignore_index=True)
base['toss_won'] = (base['toss_winner'] == base['team']).astype(int)
base = base.sort_values(['match_date', 'match_id']).reset_index(drop=True)

print(base.shape)
print(base.head())

(2338, 12)
   match_id  season match_date                                       venue  \
0    335982    2008 2008-04-18                       M Chinnaswamy Stadium   
1    335982    2008 2008-04-18                       M Chinnaswamy Stadium   
2    335983    2008 2008-04-19  Punjab Cricket Association Stadium, Mohali   
3    335983    2008 2008-04-19  Punjab Cricket Association Stadium, Mohali   
4    335984    2008 2008-04-19                            Feroz Shah Kotla   

                         team1                  team2  \
0  Royal Challengers Bangalore  Kolkata Knight Riders   
1  Royal Challengers Bangalore  Kolkata Knight Riders   
2                 Punjab Kings    Chennai Super Kings   
3                 Punjab Kings    Chennai Super Kings   
4               Delhi Capitals       Rajasthan Royals   

                   toss_winner           match_winner  \
0  Royal Challengers Bangalore  Kolkata Knight Riders   
1  Royal Challengers Bangalore  Kolkata Knight Riders   
2     

# Feature 1 — Team form (win % in last 5 matches)

In [3]:
# Sort by date to ensure correct order
base = base.sort_values(['match_date', 'match_id']).reset_index(drop=True)

def rolling_win_pct(df, window=5):
    results = []
    for team, group in df.groupby('team'):
        group = group.sort_values('match_date').copy()
        # shift(1) so we don't include current match
        group['form'] = group['won'].shift(1).rolling(window, min_periods=1).mean()
        results.append(group)
    return pd.concat(results).sort_values(['match_date', 'match_id'])

base = rolling_win_pct(base)
print(base[['team', 'match_date', 'won', 'form']].head(20))

                           team match_date  won  form
1         Kolkata Knight Riders 2008-04-18    1   NaN
0   Royal Challengers Bangalore 2008-04-18    0   NaN
3           Chennai Super Kings 2008-04-19    1   NaN
2                  Punjab Kings 2008-04-19    0   NaN
4                Delhi Capitals 2008-04-19    1   NaN
5              Rajasthan Royals 2008-04-19    0   NaN
6                Mumbai Indians 2008-04-20    0   NaN
7   Royal Challengers Bangalore 2008-04-20    1   0.0
8         Kolkata Knight Riders 2008-04-20    1   1.0
9           Sunrisers Hyderabad 2008-04-20    0   NaN
11                 Punjab Kings 2008-04-21    0   0.0
10             Rajasthan Royals 2008-04-21    1   0.0
13               Delhi Capitals 2008-04-22    1   1.0
12          Sunrisers Hyderabad 2008-04-22    0   0.0
14          Chennai Super Kings 2008-04-23    1   1.0
15               Mumbai Indians 2008-04-23    0   0.0
17             Rajasthan Royals 2008-04-24    1   0.5
16          Sunrisers Hydera

## Feature 2 — Head to head win %

In [4]:
def h2h_win_pct(df):
    results = []
    for (team, opponent), group in df.groupby(['team', 'opponent']):
        group = group.sort_values('match_date').copy()
        group['h2h_win_pct'] = group['won'].shift(1).expanding().mean()
        results.append(group)
    return pd.concat(results).sort_values(['match_date', 'match_id'])

base = h2h_win_pct(base)
print(base[['team', 'opponent', 'match_date', 'won', 'h2h_win_pct']].head(20))

                           team                     opponent match_date  won  \
1         Kolkata Knight Riders  Royal Challengers Bangalore 2008-04-18    1   
0   Royal Challengers Bangalore        Kolkata Knight Riders 2008-04-18    0   
3           Chennai Super Kings                 Punjab Kings 2008-04-19    1   
2                  Punjab Kings          Chennai Super Kings 2008-04-19    0   
4                Delhi Capitals             Rajasthan Royals 2008-04-19    1   
5              Rajasthan Royals               Delhi Capitals 2008-04-19    0   
6                Mumbai Indians  Royal Challengers Bangalore 2008-04-20    0   
7   Royal Challengers Bangalore               Mumbai Indians 2008-04-20    1   
8         Kolkata Knight Riders          Sunrisers Hyderabad 2008-04-20    1   
9           Sunrisers Hyderabad        Kolkata Knight Riders 2008-04-20    0   
11                 Punjab Kings             Rajasthan Royals 2008-04-21    0   
10             Rajasthan Royals         

# Feature 3 — Venue win %

In [6]:
def venue_win_pct(df):
    results = []
    for (team, venue), group in df.groupby(['team', 'venue']):
        group = group.sort_values('match_date').copy()
        group['venue_win_pct'] = group['won'].shift(1).expanding().mean()
        results.append(group)
    return pd.concat(results).sort_values(['match_date', 'match_id'])

base = venue_win_pct(base)
print(base[base['venue_win_pct'].notna()][['team', 'venue', 'match_date', 'won', 'venue_win_pct']].head(10))

                           team                                       venue  \
16          Sunrisers Hyderabad   Rajiv Gandhi International Stadium, Uppal   
18                 Punjab Kings  Punjab Cricket Association Stadium, Mohali   
20  Royal Challengers Bangalore                       M Chinnaswamy Stadium   
22          Chennai Super Kings             MA Chidambaram Stadium, Chepauk   
26                 Punjab Kings  Punjab Cricket Association Stadium, Mohali   
28  Royal Challengers Bangalore                       M Chinnaswamy Stadium   
30        Kolkata Knight Riders                                Eden Gardens   
32               Delhi Capitals                            Feroz Shah Kotla   
34          Sunrisers Hyderabad   Rajiv Gandhi International Stadium, Uppal   
36             Rajasthan Royals                      Sawai Mansingh Stadium   

   match_date  won  venue_win_pct  
16 2008-04-24    0            0.0  
18 2008-04-25    1            0.0  
20 2008-04-26    0    

# Feature 5 — Fill NaN values and prepare final ML dataframe

In [8]:
# Fill NaN with 0.5 (neutral — no prior info)
base['form'] = base['form'].fillna(0.5)
base['h2h_win_pct'] = base['h2h_win_pct'].fillna(0.5)
base['venue_win_pct'] = base['venue_win_pct'].fillna(0.5)

# Encode team and opponent
base['team_enc'] = base['team'].astype('category').cat.codes
base['opponent_enc'] = base['opponent'].astype('category').cat.codes

# Final feature set
features = ['team_enc', 'opponent_enc', 'toss_won', 'form', 'h2h_win_pct', 'venue_win_pct', 'season']
target = 'won'

X = base[features]
y = base[target]

print(X.shape)
print(X.isnull().sum())
print(y.value_counts())

(2338, 7)
team_enc         0
opponent_enc     0
toss_won         0
form             0
h2h_win_pct      0
venue_win_pct    0
season           0
dtype: int64
won
1    1169
0    1169
Name: count, dtype: int64


# Model Training

In [11]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

# Time-based split — important for sports data, no future leakage
# Use last 2 seasons as test set
test_seasons = [2024, 2025]
train_idx = base[~base['season'].isin(test_seasons)].index
test_idx = base[base['season'].isin(test_seasons)].index

X_train, X_test = X.loc[train_idx], X.loc[test_idx]
y_train, y_test = y.loc[train_idx], y.loc[test_idx]

print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")

# Train 3 models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, proba)
    results[name] = {'accuracy': acc, 'roc_auc': auc}
    print(f"\n{name}")
    print(f"Accuracy: {acc:.4f} | ROC-AUC: {auc:.4f}")
    print(classification_report(y_test, preds))

Train size: 2048 | Test size: 290

Logistic Regression
Accuracy: 0.4483 | ROC-AUC: 0.4601
              precision    recall  f1-score   support

           0       0.45      0.44      0.44       145
           1       0.45      0.46      0.45       145

    accuracy                           0.45       290
   macro avg       0.45      0.45      0.45       290
weighted avg       0.45      0.45      0.45       290


Random Forest
Accuracy: 0.4690 | ROC-AUC: 0.4873
              precision    recall  f1-score   support

           0       0.47      0.49      0.48       145
           1       0.47      0.45      0.46       145

    accuracy                           0.47       290
   macro avg       0.47      0.47      0.47       290
weighted avg       0.47      0.47      0.47       290


XGBoost
Accuracy: 0.4897 | ROC-AUC: 0.4885
              precision    recall  f1-score   support

           0       0.49      0.50      0.49       145
           1       0.49      0.48      0.49       145

In [10]:
!pip install xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
    --------------------------------------- 1.3/101.7 MB 5.3 MB/s eta 0:00:19
   - -------------------------------------- 2.6/101.7 MB 5.8 MB/s eta 0:00:18
   - -------------------------------------- 3.9/101.7 MB 5.8 MB/s eta 0:00:17
   -- ------------------------------------- 5.2/101.7 MB 5.7 MB/s eta 0:00:17
   -- ------------------------------------- 6.3/101.7 MB 5.6 MB/s eta 0:00:18
   -- ------------------------------------- 7.6/101.7 MB 5.6 MB/s eta 0:00:17
   --- ------------------------------------ 8.7/101.7 MB 5.7 MB/s eta 0:00:17
   --- ------------------------------------ 10.0/101.7 MB 5.7 MB/s eta 0:00:17
   ---- ----------------------------------- 11.3/101.7 MB 5.7 MB/s eta 0:00:16
   ---- ----------------------------------- 12.6/101.7 MB 5.8 MB/s eta 0:00:16
   ----- ---------------------------------- 13.9/101.7 MB 5.8 MB/s eta 0:00

In [12]:
# Feature 6 — Overall win rate of team up to that point (historical strength)
def overall_win_rate(df):
    results = []
    for team, group in df.groupby('team'):
        group = group.sort_values('match_date').copy()
        group['overall_win_rate'] = group['won'].shift(1).expanding().mean()
        results.append(group)
    return pd.concat(results).sort_values(['match_date', 'match_id'])

base = overall_win_rate(base)

# Feature 7 — Season win rate (how is team doing THIS season so far)
def season_win_rate(df):
    results = []
    for (team, season), group in df.groupby(['team', 'season']):
        group = group.sort_values('match_date').copy()
        group['season_win_rate'] = group['won'].shift(1).expanding().mean()
        results.append(group)
    return pd.concat(results).sort_values(['match_date', 'match_id'])

base = season_win_rate(base)

# Fill NaN
base['overall_win_rate'] = base['overall_win_rate'].fillna(0.5)
base['season_win_rate'] = base['season_win_rate'].fillna(0.5)

print(base[['team', 'season', 'match_date', 'overall_win_rate', 'season_win_rate']].head(20))

                           team  season match_date  overall_win_rate  \
1         Kolkata Knight Riders    2008 2008-04-18               0.5   
0   Royal Challengers Bangalore    2008 2008-04-18               0.5   
3           Chennai Super Kings    2008 2008-04-19               0.5   
2                  Punjab Kings    2008 2008-04-19               0.5   
4                Delhi Capitals    2008 2008-04-19               0.5   
5              Rajasthan Royals    2008 2008-04-19               0.5   
6                Mumbai Indians    2008 2008-04-20               0.5   
7   Royal Challengers Bangalore    2008 2008-04-20               0.0   
8         Kolkata Knight Riders    2008 2008-04-20               1.0   
9           Sunrisers Hyderabad    2008 2008-04-20               0.5   
11                 Punjab Kings    2008 2008-04-21               0.0   
10             Rajasthan Royals    2008 2008-04-21               0.0   
13               Delhi Capitals    2008 2008-04-22              

In [13]:
features = ['team_enc', 'opponent_enc', 'toss_won', 'form', 
            'h2h_win_pct', 'venue_win_pct', 'season',
            'overall_win_rate', 'season_win_rate']

X = base[features]
y = base[target]

X_train, X_test = X.loc[train_idx], X.loc[test_idx]
y_train, y_test = y.loc[train_idx], y.loc[test_idx]

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, proba)
    print(f"{name} — Accuracy: {acc:.4f} | ROC-AUC: {auc:.4f}")

Logistic Regression — Accuracy: 0.4276 | ROC-AUC: 0.4516
Random Forest — Accuracy: 0.5103 | ROC-AUC: 0.5038
XGBoost — Accuracy: 0.4966 | ROC-AUC: 0.4905


In [14]:
# Feature importance from Random Forest
import pandas as pd
import matplotlib.pyplot as plt

feat_imp = pd.Series(
    models['Random Forest'].feature_importances_,
    index=features
).sort_values(ascending=True)

print(feat_imp)

toss_won            0.032092
form                0.067573
team_enc            0.079208
opponent_enc        0.101661
venue_win_pct       0.114665
season              0.116858
season_win_rate     0.140339
h2h_win_pct         0.145503
overall_win_rate    0.202102
dtype: float64


In [15]:
print(deliveries['innings'].unique())
print(deliveries.columns.tolist())

[1 2 3 4 5 6]
['season_id', 'match_id', 'batter', 'bowler', 'non_striker', 'team_batting', 'team_bowling', 'over_number', 'ball_number', 'batter_runs', 'extras', 'total_runs', 'batsman_type', 'bowler_type', 'player_out', 'fielders_involved', 'is_wicket', 'is_wide_ball', 'is_no_ball', 'is_leg_bye', 'is_bye', 'is_penalty', 'wide_ball_runs', 'no_ball_runs', 'leg_bye_runs', 'bye_runs', 'penalty_runs', 'wicket_kind', 'is_super_over', 'innings']
